# Train the Aircraft Vision model (free Colab GPU)

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. It downloads two photo collections, then trains. Allow about **2 hours** in total, and keep the tab open:
   - FGVC-Aircraft: 100 types, 10,000 photos
   - Wikimedia Commons: free-to-reuse photos of modern types (787, A350, A220, 737 MAX, A320neo…) plus extra photos of common airliners
3. At the end it downloads `aircraft_model.zip`. Upload the two files inside it to `android/app/src/main/assets/` on GitHub.

Mounting Google Drive (optional cell below) keeps the photos and progress if Colab disconnects.

In [ ]:
!git clone https://github.com/Themagicianleo/aircraft-vision.git
%cd aircraft-vision/training
!pip -q install timm onnx onnxruntime onnxscript
!nvidia-smi --query-gpu=name,memory.total --format=csv

Optional: keep data and checkpoints on Google Drive so a disconnect doesn't lose progress.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/aircraft-vision'
else:
    WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True); print(WORK)

## Download photos from Wikimedia Commons
Which types to fetch is set in `training/commons_types.csv`. Category names that don't exist on Commons are reported with suggestions and skipped, so the run still works.
If the download is interrupted, run this cell again and it continues where it stopped.

In [ ]:
!python fetch_commons.py --out {WORK}/data/commons --per-class 250
!echo "Photos downloaded:" && find {WORK}/data/commons -name '*.jpg' | wc -l

## Train
Uses FGVC-Aircraft plus the Commons photos. If Colab disconnects, add `--resume` to continue (needs Google Drive mounted).

In [ ]:
!python train.py --data-root {WORK}/data --extra-data {WORK}/data/commons --out {WORK}/runs/effb0 --workers 2 --epochs 40 --img-size 320

## Export to ONNX (fp32 + int8) and measure accuracy with the app's exact preprocessing

In [ ]:
!python export.py --checkpoint {WORK}/runs/effb0/last.pt --out {WORK}/exported --int8 --calib-dir {WORK}/data/fgvc_prepared/images
!python evaluate.py --onnx {WORK}/exported/aircraft.onnx --data-root {WORK}/data --extra-data {WORK}/data/commons
!python evaluate.py --onnx {WORK}/exported/aircraft_int8.onnx --labels {WORK}/exported/labels.json --data-root {WORK}/data --extra-data {WORK}/data/commons

## Download
Pick the int8 model if its accuracy above is within ~1% of fp32 (it's 4× smaller). Otherwise keep fp32.

In [ ]:
USE_INT8 = False
import shutil, os
pkg = '/content/aircraft_model'
os.makedirs(pkg, exist_ok=True)
shutil.copy(f'{WORK}/exported/' + ('aircraft_int8.onnx' if USE_INT8 else 'aircraft.onnx'), f'{pkg}/aircraft.onnx')
shutil.copy(f'{WORK}/exported/labels.json', f'{pkg}/labels.json')
shutil.copy(f'{WORK}/data/commons/CREDITS.csv', f'{pkg}/PHOTO_CREDITS.csv')  # keep this: CC licences require crediting authors
shutil.make_archive('/content/aircraft_model', 'zip', pkg)
from google.colab import files
files.download('/content/aircraft_model.zip')